[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.4 MB/s eta 0:00:00


In [2]:
import torch
import math

In [5]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    B, S, D = Q.shape
    output=torch.zeros_like(Q)
    for i in range(0,S,block_size):
      q1=Q[:,i:min(S,i+block_size)]
      bs_q = q1.shape[1]
      row_max = torch.zeros((B,bs_q,1), float('-inf'), device=Q.device)
      row_sum = torch.zeros(B,bs_q,1,device=Q.device)
      for j in range(0,S,block_size):
        k1=K[:,j:j+block_size]
        vj = V[:,j:j+block_size]
        score=torch.bmm(q1,k1.transpose(1,2))/math.sqrt(D)
        block_max=torch.max(score,dim=1,keepdim=True).values
        new_max=torch.max(block_max,row_max)
        correction=torch.exp(row_max-new_max)
        exp_score=torch.exp(score-block_max)
        acc=correction*acc+torch.bmm(exp_score,vj)
        row_sum=row_sum*correction+ exp_scores.sum(dim=-1, keepdim=True)
        row_max=new_max
      output[:,i:ji+block_size]=acc/row_sum
    return output

In [6]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

TypeError: zeros() received an invalid combination of arguments - got (tuple, float, device=torch.device), but expected one of:
 * (tuple of ints size, *, tuple of names names, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')